In [103]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

ROOT = r"C:\Users\daksh\Documents\SPARSE_FFT\data"
SEQS = {
    "city":        "RECORD@2020-11-21_13.46.26",
    "countryside": "RECORD@2020-11-22_12.54.38",
    "highway":     "RECORD@2020-11-22_12.38.47",
}

In [104]:
def load_seq(tag):
    global SEQ, TAG, DATA, radar, frames
    TAG  = tag
    SEQ  = SEQS[tag]
    DATA = os.path.join(ROOT, SEQ, SEQ)
    log_path = os.path.join(DATA, SEQ + "_events_log.rec")

    radar = {"radar_ch0": {}, "radar_ch1": {}, "radar_ch2": {}, "radar_ch3": {}}
    with open(log_path) as f:
        for line in f:
            pieces = line.split()
            if len(pieces) != 17 or pieces[10] not in radar:
                continue
            radar[pieces[10]][int(pieces[7])] = (int(pieces[13]), int(pieces[16]))

    frames = [n for n in radar["radar_ch3"]
              if (n + 1) in radar["radar_ch0"]
              and (n + 2) in radar["radar_ch2"]
              and (n + 3) in radar["radar_ch1"]]
    print(TAG, SEQ, len(frames))

load_seq("city")        # city: 576, countryside: 452, highway: 388 

city RECORD@2020-11-21_13.46.26 576


In [105]:
def read_block(chip, offset, datasize):
    path = os.path.join(DATA, SEQ + f"_{chip}.bin")
    with open(path, "rb") as f:
        f.seek(offset)
        x = np.fromfile(f, dtype=np.int16, count=datasize//2)
    return x

def chip_cube(x):
    c = x[0::2] + 1j * x[1::2]
    c = np.reshape(c, (512, 4, 256), order="F")
    return c.transpose((0, 2, 1))

In [106]:
# Hamming windows (frame_to_pred needs these)
N, M = 512, 256
w_range   = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(N) / (N - 1))
w_doppler = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(M) / (M - 1))

In [107]:
from scipy.signal import convolve2d

def ca_cfar(P, n_rng=9, n_dop=9, g_rng=3, g_dop=3, thr_db=4):
    pw = np.abs(P) ** 2
    mask = np.ones((2*n_rng + 1, 2*n_dop + 1))
    mask[n_rng-g_rng : n_rng+g_rng+1, n_dop-g_dop : n_dop+g_dop+1] = 0

    sums   = convolve2d(pw, mask, mode="same")
    counts = convolve2d(np.ones_like(pw), mask, mode="same")
    noise  = sums / counts
    snr    = pw / noise
    return snr > 10 ** (thr_db / 10)

In [108]:
def frame_to_pred(n, mti=False):
    keys = [("radar_ch3", n), ("radar_ch0", n+1), ("radar_ch1", n+3), ("radar_ch2", n+2)]

    cubes = []
    for chip, s in keys:
        off, size = radar[chip][s]
        cubes.append(chip_cube(read_block(chip, off, size)))   # each is (512, 256, 4)

    cube = np.concatenate(cubes, axis=2)                     # -> (512, 256, 16)
    if mti:
        cube = cube - cube.mean(axis=1, keepdims=True)
    else:
        cube = cube - cube.mean(axis=(0, 1), keepdims=True)

    rng = np.fft.fft(cube * w_range[:, None, None],   axis=0)
    rd  = np.fft.fft(rng  * w_doppler[None, :, None], axis=1)

    Pm = np.sum(np.abs(rd), axis=2)
    return Pm.reshape(512, 16, 16).sum(axis=1)

In [109]:
def frame_to_rd(n):
    keys = [("radar_ch3", n), ("radar_ch0", n+1), ("radar_ch1", n+3), ("radar_ch2", n+2)]

    cubes = []
    for chip, s in keys:
        off, size = radar[chip][s]
        cubes.append(chip_cube(read_block(chip, off, size)))

    cube = np.concatenate(cubes, axis=2)                       # (512, 256, 16)
    cube = cube - cube.mean(axis=(0, 1), keepdims=True)        # DC offset per channel

    rng = np.fft.fft(cube * w_range[:, None, None],   axis=0)
    rd  = np.fft.fft(rng  * w_doppler[None, :, None], axis=1)

    return np.sum(np.abs(rd) ** 2, axis=2)                 # power, summed over the channels -> (512, 256)

In [110]:
P = frame_to_rd(frames[0])
print(P.shape)            # must print (512, 256)

(512, 256)


In [111]:
def k_frac(P, frac=0.9):
    p = np.sort(P.ravel())[::-1]          # all 131072 bins in one list, largest first
    c = np.cumsum(p) / p.sum()            # running share of total energy: 0.0 ... 1.0
    return int(np.searchsorted(c, frac)) + 1

In [112]:
print(k_frac(np.array([50, 30, 10, 6, 4])))     # must print 3  (50+30+10 = 90 of 100)

k90 = k_frac(P)
print(k90, k90 / P.size)                       # bins holding 90%, and as a fraction of 131072

3
6929 0.05286407470703125


In [113]:
rng_ = np.random.default_rng(0)
noise = rng_.standard_normal((512, 256, 16)) + 1j * rng_.standard_normal((512, 256, 16))

rng_n = np.fft.fft(noise * w_range[:, None, None],   axis=0)
rd_n  = np.fft.fft(rng_n * w_doppler[None, :, None], axis=1)
P_noise = np.sum(np.abs(rd_n) ** 2, axis=2)           # (512, 256), same as frame_to_rd

k90_n = k_frac(P_noise)
print(k90_n, k90_n / P_noise.size)

110919 0.8462448120117188


In [114]:
def sparsity_report(P, fracs=(0.5, 0.8, 0.9, 0.99)):
    p = np.sort(P.ravel())[::-1]
    c = np.cumsum(p) / p.sum()
    out = {}
    for f in fracs:
        k = int(np.searchsorted(c, f)) + 1      # which variable goes here?
        out[f] = k / P.size
    floor_share = np.median(P) * P.size / P.sum()
    return out, floor_share

P = frame_to_rd(frames[0])
print(SEQ, len(frames))
out, fs = sparsity_report(P)
for f, share in out.items():
    print(f, round(share, 4))
print("floor share", round(fs, 3))

RECORD@2020-11-21_13.46.26 576
0.5 0.0012
0.8 0.0165
0.9 0.0529
0.99 0.5977
floor share 0.055


In [115]:
def n_peaks(P, db):
    return int((P > np.median(P) * 10 ** (db / 10)).sum())

for tag in SEQS:
    load_seq(tag)
    P = frame_to_rd(frames[0])
    out, fs = sparsity_report(P)
    print("  K share:", {f: round(v, 4) for f, v in out.items()})
    print("  floor share:", round(fs, 3))
    print("  peaks 6 dB:", n_peaks(P, 6), " 10 dB:", n_peaks(P, 10))

print("noise  peaks 6 dB:", n_peaks(P_noise, 6), " 10 dB:", n_peaks(P_noise, 10))

city RECORD@2020-11-21_13.46.26 576
  K share: {0.5: 0.0012, 0.8: 0.0165, 0.9: 0.0529, 0.99: 0.5977}
  floor share: 0.055
  peaks 6 dB: 19611  10 dB: 11723
countryside RECORD@2020-11-22_12.54.38 452
  K share: {0.5: 0.0006, 0.8: 0.0126, 0.9: 0.1326, 0.99: 0.7512}
  floor share: 0.128
  peaks 6 dB: 8167  10 dB: 4369
highway RECORD@2020-11-22_12.38.47 388
  K share: {0.5: 0.0034, 0.8: 0.1941, 0.9: 0.4614, 0.99: 0.8475}
  floor share: 0.291
  peaks 6 dB: 8065  10 dB: 3238
noise  peaks 6 dB: 0  10 dB: 0
